# Clase 3: continuidad y estado de demostración

Clase 3 parte del estado real que dejó Clase 2. Antes de construir evidencia, comprueba qué SQLite, runs, Registry y herramientas EVM están disponibles. Este notebook **solo lee**: no inicializa, migra, siembra, registra ni promociona nada.

Los IDs cambian entre sesiones. Una SQLite sin `model_evaluations` o un Registry sin `champion` es un estado válido: vuelve al flujo de Clase 2 si necesitas preparar una demostración con datos.

In [ ]:
import json

from invoiceops.demo_state import inspect_demo_state

state = inspect_demo_state()
state_data = state.to_dict()
print(json.dumps(state_data, indent=2, sort_keys=True))

## Interpretar el resultado

| Resultado | Acción |
|---|---|
| `database.status: available` y hay `evaluation_ids` | Usa esos IDs dinámicos como entrada de las actividades posteriores de Clase 3. |
| `database.status: missing` o no hay evaluaciones | Sigue el [runbook de continuidad](../docs/class-03-continuity-runbook.md) y prepara Clase 2 en la SQLite canónica; vuelve a ejecutar esta celda. |
| `mlflow.status: not_configured` | Exporta `MLFLOW_TRACKING_URI` antes de abrir o reiniciar el kernel si necesitas inspeccionar Tracking/Registry. |
| `mlflow.status: unavailable` | Comprueba el servidor MLflow y la URI configurada. El inspector no usa un backend alternativo. |
| `mlflow.status: empty` o `champion: null` | Ejecuta el flujo explícito de Clase 2 (runs, Gate, Registry y promotion) si la actividad requiere un champion. |
| Alguna herramienta EVM en `false` | Es una precondición para tickets posteriores, no una razón para crear contratos, instalar dependencias o iniciar Anvil en este notebook. |

El directorio histórico `var/t23_5_demo/` conserva solo estado técnico de notebooks anteriores. No es una fuente de auditoría, SQLite ni MLflow para Clase 3.

## Evidence Record v1

Una evidencia solo es utilizable si la evaluación tiene `run_id` y ese run MLflow expone `dataset_version`, `feature_schema_version` y `git_commit`. La siguiente celda usa la API pública para listar cada evaluación con una causa explícita cuando no puede generar evidencia. No inventa lineage ni modifica `demo_state`.

In [ ]:
from invoiceops.evidence import list_evaluation_candidates
from invoiceops.legacy.db import _resolve_db_path

db_path = _resolve_db_path(None)
candidates = list_evaluation_candidates(db_path)
for candidate in candidates:
    print(candidate)

usable_ids = [candidate.evaluation_id for candidate in candidates if candidate.usable]
print(f"Usable evaluation IDs: {usable_ids}")

Elige un ID de `usable_ids` y ejecuta esta celda. Construye el mismo contenido lógico que `python -m invoiceops.evidence build` y persiste una única evidencia `invoice-evidence-v1` por evaluación. Al volver a ejecutarla recupera el registro existente en vez de crear una segunda evidencia.

In [ ]:
from invoiceops.evidence import (
    build_evidence_record,
    get_evidence_record,
    persist_evidence_records,
)

if not usable_ids:
    raise RuntimeError("No usable model evaluations. Complete the Clase 2 flow first.")

evaluation_id = usable_ids[0]  # Replace with another dynamic ID when teaching multiple cases.
record = get_evidence_record(db_path, evaluation_id)
if record is None:
    record = build_evidence_record(db_path, evaluation_id)
    persist_evidence_records(db_path, [record])

record.to_dict()